# Deep Learning Architectures

## Loading dataset

- MNIST dataset --> 70,000 samples of handwritten digits in grayscale image [28 X 28 pixels].
- x_train --> images
- y_train --> digit class (0-9)

In [6]:
from tensorflow.keras import datasets

(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
# Normalize pixel values (0–255 → 0–1)
x_train = x_train[..., None] / 255.0
x_test = x_test[..., None] / 255.0

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape) 

(60000, 28, 28, 1) (60000,) (10000, 28, 28, 1) (10000,)


## CNN

**Key Notes**:
- Input shape --> should match with the shape of the input sample
- There are lot of hyperparameters involved in **Conv2D** layer,
  - filters --> these can be thought of as number of output channels (or feature maps).
  - kernel_size --> larger kernels can capture broader patterns, while smaller capture local patterns more.
  - strides --> step size when scanning the kernel over the input. Higher stride leads to smaller output feature map.
  - padding --> 'same' keeps output size same as input, 'valid' reduces it.
  - activation
  - Each filter (of size 3) has --> 3x3*1(num of channel) + 1 bias = 10 parameters. For 32 filters, there will be 320 parameters.
- In **MaxPooling()**
  - pool_size --> how much down sampling is needed
  - strides
  - padding --> whether to pad the borders or not
- In **Dense()** layer
  - units
  - activation
  - kernel_initializer --> weight initialization. This affects convergence.
  - kernel_regularizer
- Following are **training hyperparameters**,
  - optimizer, learning rate, loss function, batch size, epochs, validation split

In [2]:
from tensorflow.keras import layers, models, Input

inputs = Input(shape=(28, 28, 1))             # each input sample is 28x28 pixel, and gray scale (i.e., 1 channel)
x = layers.Conv2D(32, 3, activation='relu')(inputs)  # 32 different, 3x3 filters
x = layers.MaxPooling2D(pool_size=(2,2), strides=None, padding='valid')(x)
x = layers.Conv2D(64, 3, activation='relu')(x)    # 64 filters of size 3x3
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

2025-10-19 18:03:14.501722: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       495,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 515,146 (1.97 MB)

 Trainable params: 515,146 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

> - When we do model = models.Model(inputs, outputs), we build a computation graph, which defines how the data flows through the layers.
> - model.compile() tells TensorFlow how to train the graph built in the above step. It links three components,
>   - Loss function --> what the model is trying to minimize as the cost function.
>   - Optimizer --> how to update the weights in order to reduce this loss.
>   - Metrics --> what stat to track while training.

In [4]:
# train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5


2025-10-19 18:03:15.050958: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


  5/844 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.2314 - loss: 2.2101

2025-10-19 18:03:18.147544: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17981568 exceeds 10% of free system memory.
2025-10-19 18:03:18.147847: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17981568 exceeds 10% of free system memory.
2025-10-19 18:03:18.179340: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17981568 exceeds 10% of free system memory.
2025-10-19 18:03:18.181306: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17981568 exceeds 10% of free system memory.


383/844 ━━━━━━━━━━━━━━━━━━━━ 12s 28ms/step - accuracy: 0.8436 - loss: 0.5064

844/844 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.9534 - loss: 0.1493 - val_accuracy: 0.9848 - val_loss: 0.0512
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 39s 30ms/step - accuracy: 0.9863 - loss: 0.0431 - val_accuracy: 0.9857 - val_loss: 0.0445
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.9909 - loss: 0.0282 - val_accuracy: 0.9893 - val_loss: 0.0362
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 26s 30ms/step - accuracy: 0.9932 - loss: 0.0204 - val_accuracy: 0.9902 - val_loss: 0.0355
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.9943 - loss: 0.0157 - val_accuracy: 0.9900 - val_loss: 0.0410


In [5]:
# evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9898 - loss: 0.0371
Test accuracy: 0.9898


## RNN

> We can feed the image to RNN if we treat each row (or column) as a sequence step.

> - In the MNIST dataset,
>   - We have image of 28x28 pixels.
>   - We need to **treat it as 28 time steps, each of length 28 (a row of pixels)**

In [ ]:
from tensorflow.keras import models, layers, Input

inputs = Input(shape=(28,28))  # 28 time steps, each step having 28 features.
x = layers.LSTM(128)(inputs)   # create LSTM layer with 128 units
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        80,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,290 (348.79 KB)

 Trainable params: 89,290 (348.79 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [7]:
# train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

# evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/5


2025-10-19 18:32:28.201410: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


844/844 ━━━━━━━━━━━━━━━━━━━━ 25s 28ms/step - accuracy: 0.8581 - loss: 0.4358 - val_accuracy: 0.9588 - val_loss: 0.1354
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.9602 - loss: 0.1312 - val_accuracy: 0.9710 - val_loss: 0.0940
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 23s 28ms/step - accuracy: 0.9727 - loss: 0.0885 - val_accuracy: 0.9808 - val_loss: 0.0687
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.9791 - loss: 0.0688 - val_accuracy: 0.9795 - val_loss: 0.0769
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 27s 32ms/step - accuracy: 0.9824 - loss: 0.0571 - val_accuracy: 0.9842 - val_loss: 0.0578
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9797 - loss: 0.0672
Test accuracy: 0.9797


**Key Points related to LSTM**
- The LSTM processes the sequence one row at a time.
- At each time step, it updates its internal state based on the current row (28 features) and the previous hidden state.
- After processing all 28 rows, it outputs a vector of size 128 — the final hidden state.
- The output will have shape (batch_size, 128) by default.
- This is because the LSTM returns only the last hidden state unless you specify `return_sequences=True`.

## Skip connections (ResNet)

Skip connections help with,
- preventing vanishing gradient
- encourage feature reuse
- makes training deeper networks more stable

In [9]:
inputs = Input(shape=(28,28,1))
x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
skip = x  # Save the output of the first convolution layer for later use in a skip connection. This is the residual path.
x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
x = layers.Add()([x, skip])  # Add the output of the second convolution to the saved skip tensor.
                             # This is a residual connection, similar to what’s used in ResNet.
                             # This helps with gradient flow and training stability

x = layers.Flatten()(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │        320 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 28, 28,    │          0 │ conv2d_3[0][0],   │
│                     │ 32)               │            │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 25088)     │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 10)        │    250,890 │ flatten_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 260,458 (1017.41 KB)

 Trainable params: 260,458 (1017.41 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [11]:
# train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

# evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/5


2025-10-19 18:48:56.960474: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


844/844 ━━━━━━━━━━━━━━━━━━━━ 60s 70ms/step - accuracy: 0.9506 - loss: 0.1664 - val_accuracy: 0.9827 - val_loss: 0.0618
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 81s 69ms/step - accuracy: 0.9824 - loss: 0.0579 - val_accuracy: 0.9867 - val_loss: 0.0496
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 81s 68ms/step - accuracy: 0.9871 - loss: 0.0414 - val_accuracy: 0.9850 - val_loss: 0.0560
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 83s 69ms/step - accuracy: 0.9903 - loss: 0.0302 - val_accuracy: 0.9878 - val_loss: 0.0494
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 82s 69ms/step - accuracy: 0.9929 - loss: 0.0226 - val_accuracy: 0.9872 - val_loss: 0.0502
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9871 - loss: 0.0438
Test accuracy: 0.9871
